# Module 2, Lesson 2: Working with USGS Time Series Data

## Accessing and exploring real monitoring data with Python

In the previous exercise, we used the **USGS Water Data website and Excel** to explore a time series manually.

In this notebook, we will repeat the same basic workflow using Python, but we will retrieve the data **directly from USGS**. No CSV download is needed.

We will use:

**Shoal Creek at W 12th St, Austin, TX**  
**USGS Monitoring Location: USGS-08156800**

For the worked example, we will use **discharge**.

At the end, you will repeat the analysis using **stage**.

## ✅ What you will accomplish in this lesson

By the end of this notebook, you will be able to:

✅ Retrieve USGS time series data directly in Python  
✅ Identify a monitoring location and parameter code  
✅ Inspect the returned data in a Pandas DataFrame  
✅ Check the time range of the retrieved observations  
✅ Work with timestamps and sampling intervals  
✅ Calculate basic summary statistics  
✅ Plot a time series  
✅ Detect missing records from timestamp gaps  
✅ Insert missing timestamps while leaving their values blank  
✅ Compare monthly distributions with box plots

## <font color="blue"><b>PART 1: SETTING UP</b></font>

USGS provides a Python package called `dataretrieval` that connects directly to the USGS Water Data APIs.

We will use the newer `waterdata` module.

You may find older examples online that use functions such as `nwis.get_iv()` and `nwis.get_dv()`. In this class, we will use the newer `waterdata` interface.

### About `dataretrieval` and `waterdata`

[`dataretrieval`](https://doi-usgs.github.io/dataretrieval-python/) is the USGS-supported Python package for accessing USGS water data programmatically.

Within that package, the [`waterdata`](https://doi-usgs.github.io/dataretrieval-python/reference/waterdata.html) module connects to the newer USGS Water Data APIs and provides functions such as `get_continuous()` and `get_daily()`.

In this class, we will use `waterdata` so the code matches the current USGS Water Data system.

### Installing the USGS packages

The following command installs the Python packages used in this notebook:

`!pip install dataretrieval folium`

- `!` tells Colab to run a command-line command rather than Python code.
- `pip install` installs Python packages.
- `dataretrieval` is the USGS package we use to access water data.
- `folium` is used for the optional interactive map at the end.

**Note:** You may sometimes see `-q` added after `pip install`. The `-q` means **quiet** and simply reduces the amount of installation output displayed.

In [ ]:
# Install the USGS dataretrieval package

!pip install -q dataretrieval folium

In [ ]:
# Import packages

import pandas as pd
import matplotlib.pyplot as plt
from dataretrieval import waterdata

## <font color="blue"><b>PART 2: DEFINE THE MONITORING LOCATION</b></font>

USGS identifies each monitoring location with a unique ID.

USGS also uses **parameter codes** to identify the variable being measured.

For this example:

- `USGS-08156800` = Shoal Creek at W 12th St
- `00060` = discharge
- `00065` = gage height (stage)

### USGS parameter codes

USGS uses five-digit parameter codes to identify measured variables.

A searchable list of parameter codes is available here:

[USGS Parameter Code Information](https://waterdata.usgs.gov/code-dictionary)

For this example:

- `00060` = discharge
- `00065` = gage height (stage)

In [ ]:
# Define the monitoring location and parameter

site_id = 💡
parameter_code = 💡   # Discharge

print(f'Monitoring location: {site_id}')
print(f'Parameter code: {parameter_code}')

### Retrieve information about the monitoring location

Before retrieving the time series, we can ask USGS for information about the site itself.

In [ ]:
# Retrieve monitoring location information

site_info, site_metadata = waterdata.get_monitoring_locations(
    monitoring_location_id = 💡,
    skip_geometry=True
)

site_info[
    ['monitoring_location_id',
     'monitoring_location_name',
     'site_type',
     'state_name']
]

## <font color="blue"><b>PART 3: RETRIEVE THE TIME SERIES</b></font>

Now we will retrieve continuous discharge observations directly from USGS.

`get_continuous()` retrieves high-frequency sensor observations. If we do not specify a time range, USGS returns the most recent year of available observations.

This replaces the manual steps of selecting **Download data** and opening `primary-time-series.csv`.

In [ ]:
# Retrieve continuous discharge data directly from USGS

df_flow, metadata = waterdata.💡(
    monitoring_location_id=💡,
    parameter_code=💡
)

print(f'Number of records retrieved: {len(df_flow)}')

## <font color="blue"><b>PART 4: INSPECT THE DATAFRAME</b></font>

As with any new dataset, start by looking at its structure before doing any analysis.

In [ ]:
# Display the first few rows

💡.head()

In [ ]:
# Display the column names

print('Column names:')
print(df_flow.columns)

In [ ]:
# Summarize the DataFrame structure

df_flow.info()

For this exercise, the most important columns are:

- `time` = timestamp
- `value` = discharge measurement
- `unit_of_measure` = measurement units
- `approval_status` = whether the value is approved or provisional

The API also returns metadata columns that describe the monitoring location and time series.

In [ ]:
# Keep a few important columns for inspection

df_flow[['time', 'value', 'unit_of_measure', 'approval_status']].head()

## <font color="blue"><b>PART 5: WORKING WITH TIME</b></font>

In Excel, the downloaded timestamp looked like:

```text
2026-01-06 03:30:00+00:00
```

and we had to convert it before Excel could use it as a date and time.

When we retrieve the data directly with `waterdata`, the `time` column is already stored as a Pandas datetime.

The `+00:00` indicates that the timestamp is expressed in **UTC**.

In [ ]:
# Check the timestamp data type

print(f'Timestamp data type: {df_flow['time'].dtype}')

💡 **Notice:** Python understands both the date/time and the UTC time-zone information. We do not need the Excel `LEFT()` and `VALUE()` steps.

### Sort observations from old to new

In [ ]:
# Sort observations from earliest to latest
df_flow = df_flow.💡('time')

# Reset the row numbers
df_flow = df_flow.reset_index(drop=True)

df_flow[['time', 'value']].head()

In [ ]:
df_flow[['time', 'value']].tail()

### Check the time range of the retrieved data

Before analyzing the time series, check the first and last timestamps returned by USGS.

In [ ]:
# Check the first and last timestamps in the retrieved dataset

print(f'First timestamp: {df_flow['time'].💡}')
print(f'Last timestamp:  {df_flow['time'].💡}')

### Selecting approved observations

USGS data may include both **provisional** and **approved** observations.

- **Provisional** data are preliminary observations that have not yet completed the USGS review process.
- **Approved** data have completed the review process.

In some periods, both provisional and approved observations may be returned for the same timestamp. This means that the dataset can contain more than one observation at the same time.

For this analysis, we will keep only the **approved** observations. This gives us one consistent time series for the remaining analysis.

In [ ]:
df_flow['approval_status'].value_counts()

In [ ]:
# Keep only approved observations
💡 = df_flow[💡['approval_status'] == '💡']

# Check the number of observations after filtering
print(f'Number of approved observations: {len(df_flow)}')

We now use this filtered DataFrame for the remainder of the analysis, including summary statistics, plotting, and checking for missing observations.

### Check the time range of the data with only approved observations


In [ ]:
print(f'First timestamp: {df_flow['time'].min()}')
print(f'Last timestamp:  {df_flow['time'].max()}')

## <font color="blue"><b>PART 6: BASIC SUMMARY STATISTICS</b></font>

We performed the same calculations manually in Excel.

Now calculate the mean, minimum, and maximum discharge.

In [ ]:
# Calculate basic statistics

mean_discharge = 💡['💡'].💡
min_discharge = df_flow['value'].💡
max_discharge = df_flow['value'].💡

print(f'Mean discharge: {mean_discharge:.2f} ft^3/s')
print(f'Minimum discharge: {min_discharge:.2f} ft^3/s')
print(f'Maximum discharge: {max_discharge:.2f} ft^3/s')

### When did the maximum occur?

`idxmax()` returns the row location of the maximum value.

In [ ]:
# Find the time of the maximum discharge

max_row = df_flow.loc[df_flow[💡].💡]

print(f'Maximum discharge: {max_row['value']:.2f} ft^3/s')
print(f'Time of maximum: {max_row['time']}')

In [ ]:
max_row

### Observations above and below the mean

A logical condition such as

```python
df['value'] > mean_discharge
```

returns `True` or `False` for every observation.

In Python, `True` counts as 1, so `.sum()` gives the number of observations that meet the condition.

In [ ]:
# Count observations above and below the mean

above_mean = (df_flow['value'] > 💡).💡
below_mean = (df_flow['value'] < 💡).💡

print(f'Observations above the mean: {above_mean}')
print(f'Observations below the mean: {below_mean}')

## <font color="blue"><b>PART 7: TIME SERIES PLOT</b></font>

Create the same basic time series plot that we created in Excel.

In [ ]:
# Plot discharge as a time series

plt.figure(figsize=(10, 4))
plt.plot(💡, 💡)

plt.gcf().autofmt_xdate()
plt.title('Shoal Creek Discharge')
plt.xlabel('Time')
plt.ylabel('Discharge (ft^3/s)')
plt.grid(True)

plt.show()

### Compare with the USGS website

The USGS website uses a **custom nonlinear y-axis** for its discharge graph. This expands the lower discharge values while compressing the higher values.

The plot above uses a standard linear y-axis.

Ask:

1. What features are easy to see on the linear plot?
2. What features are easier to see on the USGS plot?
3. Why can the same data look different depending on the axis scale?

## <font color="blue"><b>PART 8: CHECK FOR MISSING RECORDS</b></font>

In the downloaded CSV, missing data did not necessarily appear as blank cells or `NaN`.

Instead, entire **timestamps were missing**.

The Shoal Creek time series is expected to contain observations every **5 minutes**. We can therefore look at the time difference between consecutive records.

In [ ]:
# Calculate the time between consecutive observations

df_flow['time_gap'] = df_flow['time'].💡

df_flow[['time', 'time_gap']].head(10)

A normal observation should have a time difference of:

```text
0 days 00:05:00
```

Any interval longer than 5 minutes indicates that one or more expected records are missing.

In [ ]:
# Find time gaps larger than 5 minutes

gaps = df_flow[df_flow['time_gap'] > pd.Timedelta(minutes=💡)]

print(f'Number of gaps larger than 5 minutes: {len(gaps)}')

gaps[['time', 'time_gap']].head()

### Estimate the number of missing observations

This follows the same logic as the Excel equation:

```text
(time between records / expected 5-minute interval) - 1
```

We subtract 1 because the observation at the end of the interval is present.

In [ ]:
# Estimate the number of missing observations in each gap

df_flow['missing_records'] = (
    df_flow['time_gap'] / pd.Timedelta(minutes=5) - 1
).clip(lower=0).fillna(0).round().astype(int)

missing = df_flow[df_flow['missing_records'] > 0]

print(f'Total estimated missing observations: {df_flow['missing_records'].💡}')

missing[['time', 'time_gap', 'missing_records']].head()

### Insert the missing timestamps

So far, we identified where timestamps are missing.

We can also create a complete 5-minute time index and insert those missing timestamps into the DataFrame. The new rows are added, but the measurement values are left blank (`NaN`) for now.

This is useful because the time series now has a regular 5-minute structure without pretending that we know the missing values.

In [ ]:
# Create a complete 5-minute time index

full_time = pd.date_range(
    start=df_flow['time'].💡,
    end=df_flow['time'].💡,
    freq='💡'
)

# Reindex the DataFrame so missing timestamps are inserted
# Measurement values remain blank (NaN)

df_flow_💡 = (
    df_flow.set_index('time')
      .reindex(full_time)
      .rename_axis('time')
      .reset_index()
)

print(f'Original number of rows: {len(df_flow)}')
print(f'Rows after inserting missing timestamps: {len(df_flow_complete)}')

df_flow_complete.head()

💡 **Important:** A dataset can contain no blank values and still have missing data. In time series data, always check whether the expected timestamps are present.

## <font color="blue"><b>PART 9: TIMESERIES PLOTS</b></font>

Let's make some time series plots:
1. The entier period
2. Selected start and end dates


In [ ]:
# Plot discharge as a time series

plt.figure(figsize=(10, 4))
plt.plot(💡['time'], 💡['value'])

plt.gcf().autofmt_xdate()
plt.title('Shoal Creek Discharge')
plt.xlabel('Time')
plt.ylabel('Discharge (ft^3/s)')
plt.grid(True)

plt.show()

### **Selecting a date range**

The full time series provides an overview of the data.

In many analyses, we are interested in a specific time period, such as a day, a week, or a month. We can create a subset of the data by selecting a date range.

Lets select the seven-day period beginning at midnight on **May 15, 2026** and ending immediately before midnight on **June 15, 2026**.

In [ ]:
# Define start and end dates

start_date = '2026-💡-15'
end_date = '2026-💡-15'

💡 = df_flow_complete[
    (df_flow_complete['time'] >= start_date) &
    (df_flow_complete['time'] < end_date)
]

print(f'Number of records in subset: {len(df_subset_flow)}')


💡 **Check** How many data records you expect in an hourly dataset covering the start and end dates?

In [ ]:
# Plot a selected time period

plt.figure(figsize=(10, 4))
plt.plot(💡['time'], 💡['value'])

plt.gcf().autofmt_xdate()
plt.title('Shoal Creek Discharge')
plt.xlabel('Time')
plt.ylabel('Discharge (ft^3/s)')
plt.grid(True)

plt.show()


## <font color="blue"><b>PART 10: MONTHLY SUMMARY</b></font>

A time series plot shows when changes occurred.

A box plot provides a compact way to compare the distribution of discharge among months.

In [ ]:
# Create a month column

df_flow_complete['month'] = df_flow_complete['time'].dt.month_name().str[:3]

month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

df_flow_complete['month'] = pd.Categorical(
    df_flow_complete['month'],
    categories=month_order,
    ordered=True
)

df_flow_complete[['time', 'value', 'month']].head()

In [ ]:
# Boxplot of discharge by month

df_flow_complete.boxplot(column='value', by='month', figsize=(10, 4))

plt.title('Shoal Creek Discharge by Month')
plt.suptitle('')
plt.xlabel('Month')
plt.ylabel('Discharge (ft^3/s)')
plt.grid(True)

plt.show()

### Box plot **without displaying outlier** points

By default, a box plot shows individual points beyond the whiskers. For this discharge dataset, there can be many of them.

We can hide those individual outlier markers while keeping the box, median, and whiskers:

```python
showfliers=False
```

This does **not** remove the observations from the data. It only changes how the box plot is displayed.

In [ ]:
# Boxplot of discharge by month without showing individual outlier points

df_flow_complete.boxplot(
    column='value',
    by='month',
    figsize=(10, 4),
    showfliers=💡
)

plt.title('Shoal Creek Discharge by Month')
plt.suptitle('')
plt.xlabel('Month')
plt.ylabel('Discharge (ft^3/s)')
plt.grid(True)

plt.show()

## <font color="blue"><b>BONUS: MAKE INTERACTIVE PLOTS</b></font>

In [ ]:
import plotly.express as px

fig = px.line(df_subset_flow, x='time', y='value', title='Shoal Creek Discharge')
fig.update_layout(xaxis_title='Time', yaxis_title='Discharge (ft^3/s)')
fig.show()

# save the html
# figure_file = os.path.join(figures_folder, 'fig1_timeseries.html')
fig.write_html('fig1_timeseries.html')

## <font color="blue"><b>BONUS: MAP THE MONITORING LOCATION</b></font>

The Water Data API can also return spatial information about a monitoring location.

This is not needed for the time series analysis, but it shows that the same USGS service can provide both **data** and **location information**.

In [ ]:
# BONUS: Retrieve the monitoring location with its geometry

site_map, _ = waterdata.get_monitoring_locations(
    monitoring_location_id=site_id
)

site_map[['monitoring_location_id', 'monitoring_location_name', 'geometry']]

In [ ]:
# BONUS: Create an interactive map

import folium

longitude = site_map.geometry.x.iloc[0]
latitude = site_map.geometry.y.iloc[0]
site_name = site_map['monitoring_location_name'].iloc[0]

m = folium.Map(
    location=[latitude, longitude],
    zoom_start=14
)

folium.Marker(
    [latitude, longitude],
    popup=f'{site_name}<br>{site_id}'
).add_to(m)


m.save('shoal_creek_map.html')

m


💡This interactive visualization can be shared outside the notebook. Because we did not mount Google Drive in this session, the HTML file is saved temporarily in Colab's `/content` folder. Files stored in `/content` will disappear when the Colab session ends, so download the file to your computer if you want to keep it.

1. Click the **folder icon** on the left.
2. Find `fig1_timeseries.html` and `shoal_creek_map.html`
3. Click the **three dots** next to the file.
4. Select **Download**.

Or we can use the following code to download it from Colab's temporary /content storage to your computer.

## We can also download the .html files with code

In [ ]:
from google.colab import files

files.download('fig1_timeseries.html')


In [ ]:
files.download('shoal_creek_map.html')

## <font color="blue"><b>PART 11: PRACTICE EXERCISES</b></font>

Now repeat the workflow using **stage** instead of discharge.

USGS parameter code:

```text
00065 = Gage height (stage)
```

## 🧩 Exercise 1: Retrieve stage data

Create a DataFrame called `stage_df`.

Use:

- monitoring location `USGS-08156800`
- parameter code `00065`
- `waterdata.get_continuous()`

Do not download a file from the USGS website. Retrieve the observations directly from the API.

Then:

1. Keep only Approved observations
2. Sort the observations from earliest to latest.
3. Display the first five rows.
4. Print the number of observations.

In [ ]:
# Exercise 1 solution

stage_df, stage_metadata = waterdata.get_continuous(
    monitoring_location_id='USGS-08156800',
    parameter_code='00065'
)

# Keep only approved observations
stage_df = 💡

# Sort observations from earliest to latest
stage_df = 💡

print(f'Number of observations: {len(stage_df)}')

stage_df.head()


## 🧩 Exercise 2: Summarize stage

Calculate and print:

- mean stage
- minimum stage
- maximum stage
- date and time of the maximum stage
- number of observations above the mean
- number of observations below the mean

In [ ]:
# Exercise 2 solution

mean_stage = 💡
min_stage = 💡
max_stage = 💡

max_row = 💡

above_mean = 💡
below_mean = 💡

print(f'Mean stage: {mean_stage:.2f} ft')
print(f'Minimum stage: {min_stage:.2f} ft')
print(f'Maximum stage: {max_stage:.2f} ft')
print(f'Time of maximum stage: {max_row["time"]}')
print(f'Observations above the mean: {above_mean}')
print(f'Observations below the mean: {below_mean}')


## 🧩 Exercise 3: Plot the stage time series

Create a time series plot of stage.

Include:

- figure size of approximately `(10, 4)`
- timestamp on the x-axis
- stage on the y-axis
- descriptive title
- units
- grid lines

In [ ]:
# Exercise 3 solution

plt.figure(figsize=(10, 4))

💡

plt.gcf().autofmt_xdate()
plt.title('💡')
plt.xlabel('💡')
plt.ylabel('💡 (ft)')
plt.grid(True)

plt.show()


## 🧩 Exercise 4: Check for missing stage records

The expected sampling interval is 5 minutes.

1. Calculate the time difference between consecutive observations.
2. Identify gaps larger than 5 minutes.
3. Estimate the total number of missing observations.

Compare your result with the discharge analysis.

In [ ]:
# Exercise 4 solution

# Calculate time gaps
💡

# Identify gaps larger than 5 minutes
💡

print(f'Number of gaps larger than 5 minutes: {len(stage_gaps)}')

# Estimate the number of missing observations
💡


print(
    f'Total estimated missing observations: '
    f'{stage_df["missing_records"].sum()}'
)

stage_df.loc[
    stage_df['missing_records'] > 0,
    ['time', 'time_gap', 'missing_records']
].head()


## 🧩 Exercise 5: Add the missing timestamps

The expected sampling interval is 5 minutes.

1. Check then number of observations before and after
2. Check the number of missing observations before and after


In [ ]:
print(f'First timestamp: {stage_df['time'].min()}')
print(f'Last timestamp:  {stage_df['time'].max()}')

In [ ]:
full_time = 💡

# Reindex the DataFrame so missing timestamps are inserted
# Measurement values remain blank (NaN)

stage_df_complete = 💡

print(f'Original number of rows: {len(stage_df)}')
print(f'Rows after inserting missing timestamps: {len(stage_df_complete)}')

In [ ]:
# Calculate time gaps
💡

# Identify gaps larger than 5 minutes
💡

print(f'Number of gaps larger than 5 minutes: {len(stage_gaps)}')

# Estimate the number of missing observations

💡

## 🧩 Exercise 6: Plot stage and discharge for the same time period

Lets select one month period beginning at midnight on May 15, 2026 and ending June 15, 2026.
**Remember** to work with the complete data.
1. Define start and end dates.
2. Extract a subset spanning these dates. Check that the number of observations for stage and flow is the same.
3. Plot both on the same plot. What do you observe?
4. How would you imporve the plot?


In [ ]:
# Define start and end dates

start_date = 💡
end_date = 💡

# flow
df_subset_flow = 💡

# stage
df_subset_stage  = 💡

print(f'Number of records in subset: {len(df_subset_flow)}')
print(f'Number of records in subset: {len(df_subset_stage)}')

In [ ]:
# Plot a selected time period

plt.figure(figsize=(10, 4))
💡
💡

plt.gcf().autofmt_xdate()
plt.title('Shoal Creek Discharge')
plt.xlabel('Time')
plt.ylabel('Discharge (ft^3/s)')
plt.grid(True)

plt.show()


In [ ]:
# flow

💡

In [ ]:
# stage

💡

In [ ]:
# two y-axis

fig, ax1 = plt.subplots(figsize=(10, 4))

# Discharge - LEFT axis
ax1.plot(df_subset_flow['time'], df_subset_flow['value'], color='blue')
ax1.set_xlabel('Time')
ax1.set_ylabel('Discharge (ft³/s)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.grid(True)

# Stage - RIGHT axis
ax2 = ax1.twinx()
ax2.plot(df_subset_stage['time'], df_subset_stage['value'], color='red')
ax2.set_ylabel('Stage (ft)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

fig.autofmt_xdate()
plt.title('Shoal Creek Discharge and Stage')

plt.show()

## 🧩 Exercise 7: Monthly boxplot

1. Create a month column from the timestamp.
2. Create a box plot of stage grouped by month.
3. Which months show the greatest variability?

In [ ]:
# Exercise 6 solution

💡


## <font color="blue"><b>KEY TAKEAWAYS</b></font>

You can now:

✅ Retrieve USGS time series directly from Python  
✅ Identify a monitoring location and parameter code  
✅ Inspect a USGS DataFrame  
✅ Work with UTC timestamps  
✅ Calculate basic summary statistics  
✅ Plot a time series  
✅ Detect missing records from timestamp gaps  
✅ Estimate the number of missing observations  
✅ Compare monthly distributions with a box plot  

The analysis is similar to what we did manually in Excel. The major difference is that the workflow can now be repeated for different variables, time periods, and monitoring locations without manually downloading and editing files.

## 🧩 Exercise XX: Compare two months

Create a plot that compares **January** and **July** stage observations.

You may create two subsets of `stage_df` using the month number:

```python
stage_df['time'].dt.month
```

Think about how to place the two months on a plot so the comparison is meaningful.

In [ ]:
# Exercise 5 solution

# Create January and July subsets
january = stage_df[stage_df['time'].dt.month == 1]
july = stage_df[stage_df['time'].dt.month == 7]

plt.figure(figsize=(10, 4))

plt.plot(january['time'], january['value'], label='January')
plt.plot(july['time'], july['value'], label='July')

plt.title('Shoal Creek Stage: January and July')
plt.xlabel('Time')
plt.ylabel('Stage (ft)')
plt.grid(True)
plt.legend()

plt.show()
